# Генерация лемм при помощи LLM

**Лемматизация** — приведение словоформы к начальной форме (лемме). Это одна из самых частых задач компьютерной морфологии и особенно важна для языков с богатым словоизменением (русский, чешский и т.п.), поскольку позволяет сводить сложные словоформы к базовым и упрощает дальнейший анализ текста.

Простейший подход опирается на наблюдение, что в русском словоизменение происходит в основном в конце слова. Тогда преобразование словоформы можно описать **парой (удаляемый суффикс, добавляемый суффикс)**: `играет → играть = (ет, ть)`. Для сложных случаев — **четвёркой (префикс−, префикс+, суффикс−, суффикс+)**; например, в чешском превосходная степень: `nejstarší → starý = (nej, _, ší, ý)`.

В этих рамках задача сводится к ML-классификации: признаки берутся из аффиксов слова, обучение — на парах «словоформа — лемма» из морфологического словаря. Достаточно предсказать шаблон и проверить, что он применим к слову.

## Реальные трудности (классификация проблем)
OOV-слова — отсутствующие в словаре: композиты (электропчёлы), неологизмы (дребеденьги, хаотик).

Омонимия — одинаковое написание при разном значении: стали (глагол) vs стали (существительное).

Авторское написание — намеренные орфографические/стилистические искажения (ироглИфы).

Опечатки — непреднамеренные ошибки (напишыте).

Регистр — аббревиатуры/акронимы, которых нет в словаре или которые конфликтуют с обычной лексикой (ТО = техобслуживание vs частица то).

Упрощённое написание — намеренное отклонение от нормы (замена ё → е).

## Обзор существующих систем

Ниже — краткий обзор систем лемматизации, которые стоит знать перед тем, как переходить к генеративному подходу. Обрати внимание на три вещи: (1) как система получает морфологию — из словаря или из правил/модели; (2) как она обрабатывает слова вне словаря (OOV); (3) чем платит за качество — скоростью, памятью или ручной поддержкой словаря.

| Система | Подход | Особенности |
|---|---|---|
| **TreeTagger** | HMM + бинарное дерево решений; лемматизация по словарю (XTAG) | OOV не обрабатывает — ограничение на шумных текстах, доменной лексике и неологизмах |
| **Mystem** | Грамматический словарь (tries инвертированных основ + trie суффиксов), вероятности по longest-suffix matching | Без явных catch-all правил для неизвестных слов; строилась под ограничения IR-пайплайнов |
| **Pymorphy2** | Гибрид: DAFSA для известных слов + пайплайн эвристик для OOV | Компактная память (~7 МБ RAM при ~400 МБ XML на диске), вероятностная оценка разбора |
| **UDPipe** | Пайплайн (сегментация → токенизация → POS → лемматизация → парсинг); MorphoDiTa на averaged perceptron + Viterbi | Обучение на CoNLL-U (Universal Dependencies); POS и лемматизатор — отдельные модели, но взаимно уточняют друг друга |
| **TNPP** | Лемматизация как seq2seq (NMT): 2-слойный bi-LSTM энкодер + LSTM декодер с attention, beam search | 1-е место по лемматизации на CoNLL 2018 Shared Task |
| **Stanza** | Ансамбль: словарный лемматизатор + нейронный seq2seq + классификатор-«голова» для шорткатов (lowercasing, identity copy) | 66 языков, полностью нейронный пайплайн; шорткаты повышают устойчивость на длинных входах (URL и т.п.) |
| **Qbic** | RuBERT + данные PyMorphy2 + BiLSTM; классификация правила (длина суффикса, подстрока) из 1000 правил | Мультизадачное обучение (тегинг, лемматизация, парсинг) |
| **Rubic / Rubic2** | Rubic: архитектура Qbic + POS-информация и пост-обработка. Rubic2: Rubic + генеративный BART-base | SOTA по разным доменам (новости, художественная литература, поэзия, вики, соцсети), но BART медленный (~707 TPS против ~1416 TPS у BERT-классификатора) → генератор применяется только к PROPN |

## Ключевой вывод

Эволюция подходов идёт от **словарных и rule-based методов** (TreeTagger, Mystem, Pymorphy2) через **нейросетевые классификаторы правил** (Qbic, Rubic) к **генеративным моделям** (seq2seq в TNPP/Stanza, BART в Rubic2). Генеративный подход даёт качество, но упирается в скорость инференса.

In [1]:
import requests
import pathlib
import random
import time
from tqdm import tqdm
from conllu import parse_incr, TokenList
import datasets
from typing import Callable
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch
import random
import numpy as np
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

In [2]:
random.seed(42)  # для воспроизводимости

In [3]:
# создадим папку ./data для данных

DATADIR = pathlib.Path.cwd() / "data"
DATADIR.mkdir(exist_ok=True)

In [4]:
# данные стянем из интернета.

# юзер агент обозначим чтобы постараться избежать рандомной блокировки
USER_AGENT = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
headers = {
    "User-Agent": USER_AGENT
}

# ссылки на raw файлы из https://github.com/UniversalDependencies/UD_Russian-SynTagRus
links = [
    "https://github.com/UniversalDependencies/UD_Russian-SynTagRus/raw/refs/heads/master/ru_syntagrus-ud-dev.conllu",
    "https://github.com/UniversalDependencies/UD_Russian-SynTagRus/raw/refs/heads/master/ru_syntagrus-ud-test.conllu",
    "https://github.com/UniversalDependencies/UD_Russian-SynTagRus/raw/refs/heads/master/ru_syntagrus-ud-train-a.conllu",
    "https://github.com/UniversalDependencies/UD_Russian-SynTagRus/raw/refs/heads/master/ru_syntagrus-ud-train-b.conllu",
    "https://github.com/UniversalDependencies/UD_Russian-SynTagRus/raw/refs/heads/master/ru_syntagrus-ud-train-c.conllu",
]

for link in links:

    cur_filepath = DATADIR / pathlib.Path(link).name

    # пропустим если уже скачали
    if cur_filepath.exists():
        print(f"{cur_filepath.name} exists at {cur_filepath}, skip download.")

    else:

        # тянем-потянем
        try:
            print(f"Downloading {cur_filepath.name} from {link}")
            response = requests.get(link, headers=headers, timeout=30)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(e)
            break

        # если все ОК, запишем всё в файл
        with open(
            cur_filepath,      # открываем файл
            "w",               # новый, на запись
            encoding="utf-8",  # в кодировке utf-8 (важно для windows)
        ) as file:
            file.write(response.text)

        delay = random.randint(100, 300) / 100  # случайная десятичная дробь между 1.00 и 3.00
        print(f"Successful. Sleep {delay}s\n")
        time.sleep(delay)  # делаем задержку между запросами чтобы не улететь в бан

ru_syntagrus-ud-dev.conllu exists at /home/kirill/sandbox/lemmatization-lesson/data/ru_syntagrus-ud-dev.conllu, skip download.
ru_syntagrus-ud-test.conllu exists at /home/kirill/sandbox/lemmatization-lesson/data/ru_syntagrus-ud-test.conllu, skip download.
ru_syntagrus-ud-train-a.conllu exists at /home/kirill/sandbox/lemmatization-lesson/data/ru_syntagrus-ud-train-a.conllu, skip download.
ru_syntagrus-ud-train-b.conllu exists at /home/kirill/sandbox/lemmatization-lesson/data/ru_syntagrus-ud-train-b.conllu, skip download.
ru_syntagrus-ud-train-c.conllu exists at /home/kirill/sandbox/lemmatization-lesson/data/ru_syntagrus-ud-train-c.conllu, skip download.


In [ ]:
# функция для получения сплитов
# мы будем тренировать несколько моделей, поэтому дублирующийся код уберем в функцию
# datadir -- папка с данными .conllu
# prepare_sample -- функция, на вход TokenList из parse_incr, на выход -- список пар вход/выход
# samples_per_split -- сколько сэмплов берем из трейн дев тест соответственно

def get_train_dev_test_data(
    datadir: pathlib.Path,
    prepare_samples: Callable[[TokenList], list[dict[str, str]]],
    samples_per_split=(100000, 50000, 50000),
):   

    train_data = []
    dev_data = []
    test_data = []

    for filename in datadir.iterdir():

        assert filename.suffix == ".conllu", f"Foreign file {filename} while iterating {datadir}."  # проверим что ничего лишнего в папке нет

        # в имени файла написано, к какому сплиту он относится 
        if "train" in filename.name:
            cur_container = train_data
        elif "dev" in filename.name:
            cur_container = dev_data
        elif "test" in filename.name:
            cur_container = test_data
        else:
            raise AssertionError(f"Check datadir: there might be foreign files there: {filename}")

        with open(
            filename,         # откроем файл
            "r",              # на чтение
            encoding="utf-8"  # в кодировке utf-8 (важно для windows)
        ) as file:
            for sentence in parse_incr(file):
                cur_container.extend(
                    prepare_samples(sentence)
                )

    # возьмем только часть данных, чтобы не училось всю жизнь
    train_data = random.sample(train_data, min(len(train_data), samples_per_split[0]))
    dev_data = random.sample(dev_data, min(len(dev_data), samples_per_split[1]))
    test_data = random.sample(test_data, min(len(test_data), samples_per_split[2]))

    return train_data, dev_data, test_data 

In [6]:
# первый вариант: форма на вход, лемма на выход
def prepare_raw_form_input(sentence: TokenList):

    result = []

    for token in sentence:

        form = token["form"]
        lemma = token["lemma"]
        assert form and lemma

        result.append(
            {"form": form, "lemma": lemma}
        )

    return result

In [7]:
train_data, dev_data, test_data = get_train_dev_test_data(DATADIR, prepare_raw_form_input)

In [8]:
# проверим пересечения сплитов

def check_intersection(train_data, dev_data, test_data):
    train_set = set([(d["form"], d["lemma"]) for d in train_data])
    dev_set   = set([(d["form"], d["lemma"]) for d in dev_data])
    test_set  = set([(d["form"], d["lemma"]) for d in test_data])

    print(f"train ∩ dev set intersection = {len(train_set & dev_set)}")
    print(f"train ∩ test set intersection= {len(train_set & test_set)}")
    print(f"dev ∩ test set intersection  = {len(dev_set & test_set)}")

In [9]:
# пересечения есть, но в нашем случае это нормально.
# некоторые слова -- частицы, предлоги, союзы, частотные глаголы и имена и тд есть везде.
# то есть в данном случае это не утечка, а свойство языка и речи.

check_intersection(train_data, dev_data, test_data )

train ∩ dev set intersection = 7028
train ∩ test set intersection= 7110
dev ∩ test set intersection  = 5126


In [10]:
dataset = datasets.DatasetDict(
    {
        "train": datasets.Dataset.from_list(train_data),
        "dev": datasets.Dataset.from_list(dev_data),
        "test": datasets.Dataset.from_list(test_data)
    }
)

In [11]:
print(dataset)
print("Пример train:", dataset["train"][0])
print("Пример dev:  ", dataset["dev"][0])
print("Пример test: ", dataset["test"][0])
print("Колонки:", dataset["train"].column_names)
print("Размеры:", {k: len(v) for k, v in dataset.items()})

DatasetDict({
    train: Dataset({
        features: ['form', 'lemma'],
        num_rows: 100000
    })
    dev: Dataset({
        features: ['form', 'lemma'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['form', 'lemma'],
        num_rows: 50000
    })
})
Пример train: {'form': 'доходов', 'lemma': 'доход'}
Пример dev:   {'form': 'института', 'lemma': 'институт'}
Пример test:  {'form': 'оказалась', 'lemma': 'оказаться'}
Колонки: ['form', 'lemma']
Размеры: {'train': 100000, 'dev': 50000, 'test': 50000}


In [12]:
MODEL_ID = "ai-forever/ruT5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 73184.29it/s]


In [13]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [13]:
MAX_LEN = 16

In [14]:
def tokenize_fn(examples, tokenizer=tokenizer):
    model_inputs = tokenizer(
        examples["form"],
        max_length=MAX_LEN,
        truncation=True,
    )
    labels = tokenizer(
        examples["lemma"],
        max_length=MAX_LEN,
        truncation=True,
    )
    # заменяем pad в labels на -100, чтобы loss не считался на паддинге
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [15]:
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)

Tokenizing: 100%|██████████| 50000/50000 [00:00<00:00, 101897.23 examples/s]


In [16]:
tokenized_dataset["train"][1234]

{'input_ids': [1240, 76, 2], 'attention_mask': [1, 1, 1], 'labels': [6732, 2]}

In [17]:
collator = DataCollatorForSeq2Seq(tokenizer, model)

In [18]:
collator(
    [tokenized_dataset["train"][i] for i in range(3)]
)

{'input_ids': tensor([[10533,     2,     0],
        [    8,     3,     2],
        [  952,     2,     0]]), 'attention_mask': tensor([[1, 1, 0],
        [1, 1, 1],
        [1, 1, 0]]), 'labels': tensor([[3363,    2, -100],
        [   8,    3,    2],
        [1677,    7,    2]]), 'decoder_input_ids': tensor([[   0, 3363,    2],
        [   0,    8,    3],
        [   0, 1677,    7]])}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="models/ruT5-lemmatizer",
    num_train_epochs=20,
    per_device_train_batch_size=128*2,
    per_device_eval_batch_size=128*4,
    learning_rate=1e-4
    warmup_steps=500,
    logging_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=True,     # важно для seq2seq: метрики считаются по generate()
    fp16=True,
    report_to="none",
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    seed=42,
)

In [15]:
# lemmatization accuracy по Ляшевской:
# отношение кол-ва верных лемм к общему количеству безотносительно регистра букв и выбора е/ё

def preproc_for_lAcc(words):
    return [word.lower().replace("ё", "е") for word in words]

def lAcc(y: list[str], y_bar: list[str]):
    if not y:
        return 0.0

    y = preproc_for_lAcc(y)
    y_bar = preproc_for_lAcc(y_bar)

    hits = 0
    for target, pred in zip(y, y_bar):
        if target == pred:
            hits += 1

    return hits / len(y)

def compute_metrics(eval_preds, tokenizer=tokenizer):
    preds, labels = eval_preds

    # preds — tuple/list; после predict_with_generate это сгенерированные id
    if isinstance(preds, tuple):
        preds = preds[0]

    # поменяем -100 на паддинг
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    return {"accuracy": lAcc(decoded_labels, decoded_preds)}

In [28]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [29]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
500,1.288550,0.097633,0.953960
1000,0.111240,0.084294,0.962560
1500,0.068018,0.079705,0.965400
2000,0.047902,0.083123,0.964240
2500,0.037599,0.084766,0.963240
3000,0.032113,0.088018,0.965060


/home/kirill/sandbox/lemmatization-lesson/.venv/lib/python3.14/site-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/home/kirill/sandbox/lemmatization-lesson/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3000, training_loss=0.2642372678120931, metrics={'train_runtime': 415.7201, 'train_samples_per_second': 1845.625, 'train_steps_per_second': 7.216, 'total_flos': 5522601049620480.0, 'train_loss': 0.2642372678120931, 'epoch': 7.672634271099744})

In [31]:
trainer.evaluate(tokenized_dataset["test"])

Training Loss,Validation Loss,Step,Accuracy
0.032113,0.078612,3000,0.966760


{'eval_loss': 0.07861214876174927, 'eval_accuracy': 0.96676}

In [ ]:
def predict(
    texts: list[str],
    model,
    tokenizer,
    batch_size: int = 32,
    verbose=True,
) -> list[str]:

    preds = []

    if verbose:
        pbar: range | tqdm[int] = tqdm(
            range(0, len(texts), batch_size),
            total=np.ceil(len(texts) / batch_size),
        )
    else:
        pbar = range(0, len(texts), batch_size)

    for i in pbar:
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            max_length=MAX_LEN,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_token_type_ids=False,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_LEN,
                num_beams=1,
                temperature=1,
            )

        cur_preds = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        preds.extend(cur_preds)

    return preds

In [69]:
examples = [
    "Он стоит очень дорого",
    "Стёпка , хочешь щец ? — Фи !",
    "напишыте мне на почту",
    "Это электропчёлы и они делают электромёд",
    "этот мужчина неописуемо серфингист"
]


In [70]:
for example in examples:
    split_example = example.split()
    lemmas = predict(
        example.split(),
        model,
        tokenizer,
        verbose=False
    )

    print(f"form       lemma")
    print(f"----------------")
    for form, lemma in zip(split_example, lemmas):
        print(f"{form:<10} {lemma}")
    print()

form       lemma
----------------
Он         он
стоит      стоять
очень      очень
дорого     дорого

form       lemma
----------------
Стёпка     Стёпка
,          ,
хочешь     хотеть
щец        щец
?          ?
—          —
Фи         Фи
!          !

form       lemma
----------------
напишыте   напишыть
мне        я
на         на
почту      почта

form       lemma
----------------
Это        это
электропчёлы электропчёла
и          и
они        они
делают     делать
электромёд электромёд

form       lemma
----------------
этот       этот
мужчина    мужчина
неописуемо неописуемо
серфингист серфингист



In [ ]:
del model
del tokenizer
del trainer

In [74]:
torch.cuda.empty_cache()

In [ ]:
# второй вариант: форма и фичи на вход, лемма на выход
def prepare_form_pos_feats_input(sentence: TokenList):

    result = []

    for token in sentence:
        form = token["form"]
        lemma = token["lemma"]
        pos = token["upos"]
        feats = token.get("feats")
        assert form and lemma and pos

        inpt = f"{form} {pos}"
        if feats:
            for k, val in feats.items():
                inpt+=f" {k}:{val}"

        result.append(
            {"form": inpt, "lemma": lemma}
        )

    return result

In [98]:
train_data, dev_data, test_data = get_train_dev_test_data(DATADIR, prepare_form_pos_feats_input)

In [ ]:
check_intersection(train_data, dev_data, test_data )

train ∩ dev set intersection = 7169
train ∩ test set intersection= 7347
dev ∩ test set intersection  = 5214


In [101]:
dataset = datasets.DatasetDict(
    {
        "train": datasets.Dataset.from_list(train_data),
        "dev": datasets.Dataset.from_list(dev_data),
        "test": datasets.Dataset.from_list(test_data)
    }
)

In [102]:
print(dataset)
print("Пример train:", dataset["train"][0])
print("Пример dev:  ", dataset["dev"][0])
print("Пример test: ", dataset["test"][0])
print("Колонки:", dataset["train"].column_names)
print("Размеры:", {k: len(v) for k, v in dataset.items()})

DatasetDict({
    train: Dataset({
        features: ['form', 'lemma'],
        num_rows: 100000
    })
    dev: Dataset({
        features: ['form', 'lemma'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['form', 'lemma'],
        num_rows: 50000
    })
})
Пример train: {'form': 'доходов NOUN Animacy:Inan Case:Gen Gender:Masc Number:Plur', 'lemma': 'доход'}
Пример dev:   {'form': 'института NOUN Animacy:Inan Case:Gen Gender:Masc Number:Sing', 'lemma': 'институт'}
Пример test:  {'form': 'оказалась VERB Aspect:Perf Gender:Fem Mood:Ind Number:Sing Tense:Past VerbForm:Fin Voice:Mid', 'lemma': 'оказаться'}
Колонки: ['form', 'lemma']
Размеры: {'train': 100000, 'dev': 50000, 'test': 50000}


In [103]:
MODEL_ID = "ai-forever/ruT5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 58883.32it/s]


In [112]:
MAX_LEN = 64

In [113]:
def tokenize_fn(examples, tokenizer=tokenizer):
    model_inputs = tokenizer(
        examples["form"],
        max_length=MAX_LEN,
        truncation=True,
    )
    labels = tokenizer(
        examples["lemma"],
        max_length=MAX_LEN,
        truncation=True,
    )
    # заменяем pad в labels на -100, чтобы loss не считался на паддинге
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [114]:
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)

Tokenizing: 100%|██████████| 50000/50000 [00:00<00:00, 82829.83 examples/s]


In [118]:
collator = DataCollatorForSeq2Seq(tokenizer, model)

In [120]:
training_args = Seq2SeqTrainingArguments(
    output_dir="models/ruT5-lemmatizer_2",
    num_train_epochs=20,
    per_device_train_batch_size=128*2,
    per_device_eval_batch_size=128*4,
    learning_rate=1e-4,
    warmup_steps=500,
    logging_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=True,     # важно для seq2seq: метрики считаются по generate()
    fp16=True,
    report_to="none",
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    seed=42,
)

In [121]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [122]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
500,1.328816,0.060780,0.975320
1000,0.069875,0.046180,0.982800
1500,0.035194,0.044015,0.985640
2000,0.022098,0.045948,0.985240
2500,0.015984,0.048202,0.986600
3000,0.012449,0.045958,0.985420
3500,0.010524,0.050116,0.987140
4000,0.008811,0.051032,0.986160
4500,0.008048,0.051313,0.986900
5000,0.006657,0.050527,0.986140


/home/kirill/sandbox/lemmatization-lesson/.venv/lib/python3.14/site-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=5000, training_loss=0.15184553079605104, metrics={'train_runtime': 2318.838, 'train_samples_per_second': 551.471, 'train_steps_per_second': 2.156, 'total_flos': 7.784396252725248e+16, 'train_loss': 0.15184553079605104, 'epoch': 12.787723785166241})

In [123]:
trainer.evaluate(tokenized_dataset["test"])

Training Loss,Validation Loss,Step,Accuracy
0.006657,0.047739,5000,0.987860


{'eval_loss': 0.04773862659931183, 'eval_accuracy': 0.98786}

In [144]:

examples = [
    "стали NOUN Animacy:Inan Case=Nom Gender=Fem Number=Plur",
    "стали VERB Aspect:Perf Mood:Ind Number:Plur Tense:Past VerbForm:Fin Voice:Mid",
]

In [146]:
for example in examples:
    split_example = [example]
    lemmas = predict(
        example,
        model,
        tokenizer,
        verbose=False
    )

    print(f"form       lemma")
    print(f"----------------")
    for form, lemma in zip(split_example, lemmas):
        print(1)
        print(f"{form.split()[0]:<10} {lemma}")
    print()

form       lemma
----------------
1
стали      сталь

form       lemma
----------------
1
стали      стать



In [149]:
model.save_pretrained("./models/lemmatizer_form_and_feats")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]


In [150]:
tokenizer.save_pretrained("./models/lemmatizer_form_and_feats")

('./models/lemmatizer_form_and_feats/tokenizer_config.json',
 './models/lemmatizer_form_and_feats/tokenizer.json')

In [39]:
del model

In [40]:
del tokenizer

In [41]:
torch.cuda.empty_cache()

In [20]:
# третий вариант: лемматизация всего предложения сразу
def prepare_whole_sentence_input(sentence: TokenList):

    result = [
        {
            "form": " ".join([token["form"] for token in sentence]),
            "lemma": " ".join([token["lemma"] for token in sentence]),
        }        
    ]

    return result

In [21]:
train_data, dev_data, test_data = get_train_dev_test_data(DATADIR, prepare_whole_sentence_input, samples_per_split=(10000, 5000, 5000))

In [22]:
# тут пересечений уже нет тк работаем с предложениями

check_intersection(train_data, dev_data, test_data )

train ∩ dev set intersection = 5
train ∩ test set intersection= 5
dev ∩ test set intersection  = 7


In [23]:
dataset = datasets.DatasetDict(
    {
        "train": datasets.Dataset.from_list(train_data),
        "dev": datasets.Dataset.from_list(dev_data),
        "test": datasets.Dataset.from_list(test_data)
    }
)

In [24]:
print(dataset)
print("Пример train:", dataset["train"][0])
print("Пример dev:  ", dataset["dev"][0])
print("Пример test: ", dataset["test"][0])
print("Колонки:", dataset["train"].column_names)
print("Размеры:", {k: len(v) for k, v in dataset.items()})

DatasetDict({
    train: Dataset({
        features: ['form', 'lemma'],
        num_rows: 10000
    })
    dev: Dataset({
        features: ['form', 'lemma'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['form', 'lemma'],
        num_rows: 5000
    })
})
Пример train: {'form': 'Им отвечали : " Живаго " .', 'lemma': 'они отвечать : " Живаго " .'}
Пример dev:   {'form': '- Экспедиции на Марс это подтверждают !', 'lemma': '- экспедиция на Марс это подтверждать !'}
Пример test:  {'form': 'Эта же компания позже распределяла доходы от трансляций .', 'lemma': 'этот же компания позже распределять доход от трансляция .'}
Колонки: ['form', 'lemma']
Размеры: {'train': 10000, 'dev': 5000, 'test': 5000}


In [42]:
MODEL_ID = "ai-forever/ruT5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 49358.15it/s]


In [43]:
MAX_LEN = 80

In [44]:
def tokenize_fn(examples, tokenizer=tokenizer):
    model_inputs = tokenizer(
        examples["form"],
        max_length=MAX_LEN,
        truncation=True,
    )
    labels = tokenizer(
        examples["lemma"],
        max_length=MAX_LEN,
        truncation=True,
    )
    # заменяем pad в labels на -100, чтобы loss не считался на паддинге
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [45]:
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)

Tokenizing: 100%|██████████| 5000/5000 [00:00<00:00, 36293.83 examples/s]


In [46]:
collator = DataCollatorForSeq2Seq(tokenizer, model)

In [48]:
training_args = Seq2SeqTrainingArguments(
    output_dir="models/ruT5-lemmatizer_3",
    num_train_epochs=20,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128*2,
    learning_rate=1e-4,
    warmup_steps=50,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    predict_with_generate=True,     # важно для seq2seq: метрики считаются по generate()
    fp16=True,
    report_to="none",
    metric_for_best_model="wer",
    greater_is_better=False,
    load_best_model_at_end=True,
    seed=42,
    generation_max_length=MAX_LEN, 
)

In [49]:
from jiwer import wer

In [50]:
def compute_metrics(eval_preds, tokenizer=tokenizer):
    preds, labels = eval_preds

    # preds — tuple/list; после predict_with_generate это сгенерированные id
    if isinstance(preds, tuple):
        preds = preds[0]

    # поменяем -100 на паддинг
    preds  = np.where(preds  >= 0, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # здесь lAcc посчитать уже не так просто тк иногда модель галлюцинирует и возвращает кол-во лемм != кол-во форм
    # поэтому используем word error rate для оценки качества

    return {"wer": wer(decoded_labels, decoded_preds)}

In [51]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [52]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
50,4.229309,0.428653,0.158068
100,0.477696,0.179136,0.063993
150,0.298242,0.133431,0.042627
200,0.213548,0.123131,0.037958
250,0.175599,0.107249,0.031242
300,0.144488,0.103476,0.031206
350,0.123464,0.099769,0.028539
400,0.114227,0.094832,0.028211
450,0.094020,0.095824,0.027135
500,0.089199,0.093273,0.027603


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.32it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1580, training_loss=0.21913462718830834, metrics={'train_runtime': 1631.4768, 'train_samples_per_second': 122.588, 'train_steps_per_second': 0.968, 'total_flos': 1.83411456159744e+16, 'train_loss': 0.21913462718830834, 'epoch': 20.0})

In [60]:
examples = [
    "Люди всё время меня спрашивают , знаю ли я Тайлера Дердэна .",
    "мама мыла раму",
    "самые прочные стали на заводе",
    "они стали гораздо способнее",
    "ошыбки неизбежны",
    "они стоят на горе",
    "они стоят сто рублей",
]

In [61]:
predict(
    examples,
    model,
    tokenizer
)

100%|██████████| 1/1.0 [00:00<00:00,  8.00it/s]


['человек весь время я спрашивать , знать ли я Тайлер Дерден .',
 'мама мыть рама .',
 'самый прочный сталь на завод',
 'они стать гораздо способный',
 'ошыбка неизбежный',
 'они стоять на гора.',
 'они стоить сто рубль.']

In [57]:
trainer.evaluate(tokenized_dataset["test"])

Training Loss,Validation Loss,Step,Wer
0.029821,0.090452,1580,0.020090


{'eval_loss': 0.0904519110918045, 'eval_wer': 0.020090323437603773}

In [62]:
model.save_pretrained("./models/lemmatizer_sentence_level")
tokenizer.save_pretrained("./models/lemmatizer_sentence_level")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]


('./models/lemmatizer_sentence_level/tokenizer_config.json',
 './models/lemmatizer_sentence_level/tokenizer.json')